In [16]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# Define the CNN architecture (same as in MNIST_CNN.py)
class CNNClassifier(nn.Module):
    def __init__(self):
        super(CNNClassifier, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(128*3*3, 128)
        self.fc2 = nn.Linear(128, 10)  # For 10 output classes (digits 0-9)
    
    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = torch.max_pool2d(x, 2)
        x = torch.relu(self.conv2(x))
        x = torch.max_pool2d(x, 2)
        x = torch.relu(self.conv3(x))
        x = torch.max_pool2d(x, 2)
        x = x.view(-1, 128*3*3)
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# Load the MNIST dataset
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
trainset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = DataLoader(trainset, batch_size=64, shuffle=True)

# Initialize the model, loss, and optimizer
model = CNNClassifier()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Train the model
num_epochs = 5
for epoch in range(num_epochs):
    for data, target in train_loader:
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()

# Save the model's state_dict (weights)
torch.save(model, "./ModelFiles/model.pt")


100%|██████████| 9.91M/9.91M [00:06<00:00, 1.44MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 111kB/s]
100%|██████████| 1.65M/1.65M [00:02<00:00, 619kB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 8.04MB/s]


In [17]:
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torch import nn

# Load the Fashion-MNIST dataset
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
mnist_testset = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)
test_loader = DataLoader(mnist_testset, batch_size=64, shuffle=False)

# Define the CNN architecture (same as the one used in MNIST_CNN.py)
class CNNClassifier(nn.Module):
    def __init__(self):
        super(CNNClassifier, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(128*3*3, 128)
        self.fc2 = nn.Linear(128, 10)
    
    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = torch.max_pool2d(x, 2)
        x = torch.relu(self.conv2(x))
        x = torch.max_pool2d(x, 2)
        x = torch.relu(self.conv3(x))
        x = torch.max_pool2d(x, 2)
        x = x.view(-1, 128*3*3)
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# Load the pre-trained model (MNIST model)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = torch.load("./ModelFiles/model.pt", weights_only=False)
model.to(device)

# Print model's state_dict (to see the parameters)
print("Model's state_dict:")
for param_tensor in model.state_dict().keys():
    print(param_tensor, "\t", model.state_dict()[param_tensor].size())

# Evaluate the model
model.eval()
correct = 0
total = 0

# Iterate over the test set (Fashion-MNIST)
with torch.no_grad():
    for i, vdata in enumerate(test_loader):
        tinputs, tlabels = vdata
        tinputs = tinputs.to(device)
        tlabels = tlabels.to(device)
        toutputs = model(tinputs)
        
        # Get the predicted class label
        _, predicted = torch.max(toutputs, 1)
        print(f"True label: {tlabels}")
        print(f"Predicted: {predicted}")
        
        total += tlabels.size(0)
        correct += (predicted == tlabels).sum().item()

accuracy = 100.0 * correct / total
print(f"The overall accuracy is {accuracy}%")


Model's state_dict:
conv1.weight 	 torch.Size([32, 1, 3, 3])
conv1.bias 	 torch.Size([32])
conv2.weight 	 torch.Size([64, 32, 3, 3])
conv2.bias 	 torch.Size([64])
conv3.weight 	 torch.Size([128, 64, 3, 3])
conv3.bias 	 torch.Size([128])
fc1.weight 	 torch.Size([128, 1152])
fc1.bias 	 torch.Size([128])
fc2.weight 	 torch.Size([10, 128])
fc2.bias 	 torch.Size([10])
True label: tensor([9, 2, 1, 1, 6, 1, 4, 6, 5, 7, 4, 5, 7, 3, 4, 1, 2, 4, 8, 0, 2, 5, 7, 9,
        1, 4, 6, 0, 9, 3, 8, 8, 3, 3, 8, 0, 7, 5, 7, 9, 6, 1, 3, 7, 6, 7, 2, 1,
        2, 2, 4, 4, 5, 8, 2, 2, 8, 4, 8, 0, 7, 7, 8, 5])
Predicted: tensor([1, 8, 1, 1, 1, 1, 1, 8, 2, 2, 4, 2, 2, 1, 2, 1, 1, 8, 2, 7, 2, 2, 2, 0,
        1, 1, 6, 1, 1, 1, 2, 5, 1, 8, 2, 7, 2, 2, 2, 1, 8, 1, 1, 2, 6, 1, 6, 1,
        3, 0, 0, 2, 2, 5, 8, 6, 0, 0, 2, 1, 2, 2, 2, 0])
True label: tensor([1, 1, 2, 3, 9, 8, 7, 0, 2, 6, 2, 3, 1, 2, 8, 4, 1, 8, 5, 9, 5, 0, 3, 2,
        0, 6, 5, 3, 6, 7, 1, 8, 0, 1, 4, 2, 3, 6, 7, 2, 7, 8, 5, 9, 9, 4, 2, 5,
     

In [20]:
# Step 1: Import necessary libraries
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision import datasets, models
import os
import matplotlib.pyplot as plt
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_transforms = transforms.Compose([
    transforms.Resize(256),  
    transforms.RandomCrop(227),  
    transforms.RandomHorizontalFlip(),  
    transforms.ToTensor(),  
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  
])

val_transforms = transforms.Compose([
    transforms.Resize(256),  
    transforms.CenterCrop(227),  
    transforms.ToTensor(),  
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  
])

data_dir = './cats_and_dogs_filtered'

train_dir = os.path.join(data_dir, 'train')
val_dir = os.path.join(data_dir, 'validation')

train_dataset = datasets.ImageFolder(train_dir, transform=train_transforms)
val_dataset = datasets.ImageFolder(val_dir, transform=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

alexnet = models.alexnet(pretrained=True)

alexnet.classifier[6] = nn.Linear(in_features=4096, out_features=2)  # Two classes: cats and dogs

alexnet = alexnet.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(alexnet.parameters(), lr=0.001)

num_epochs = 3

for epoch in range(num_epochs):
    alexnet.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for inputs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        
        outputs = alexnet(inputs)
        
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100 * correct / total
    print(f"Epoch {epoch+1}: Loss = {epoch_loss:.4f}, Accuracy = {epoch_acc:.2f}%")
    
    alexnet.eval()
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = alexnet(inputs)
            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    val_acc = 100 * val_correct / val_total
    print(f"Validation Accuracy: {val_acc:.2f}%")

torch.save(alexnet.state_dict(), 'alexnet_finetuned.pth')

Epoch 1/3: 100%|██████████| 63/63 [00:36<00:00,  1.73it/s]


Epoch 1: Loss = 0.8287, Accuracy = 50.40%
Validation Accuracy: 53.30%


Epoch 2/3: 100%|██████████| 63/63 [00:40<00:00,  1.57it/s]


Epoch 2: Loss = 0.6976, Accuracy = 50.15%
Validation Accuracy: 50.00%


Epoch 3/3: 100%|██████████| 63/63 [00:37<00:00,  1.67it/s]


Epoch 3: Loss = 0.6946, Accuracy = 48.40%
Validation Accuracy: 50.00%


In [24]:
checkpoint = {
    'epoch': epoch,
    'model_state': alexnet.state_dict(),
    'optimizer_state': optimizer.state_dict(),
    'loss': loss.item()
}
torch.save(checkpoint, './alexnet_finetuned.pth')

# To resume training from checkpoint
checkpoint = torch.load('./alexnet_finetuned.pth')
alexnet.load_state_dict(checkpoint['model_state'])
optimizer.load_state_dict(checkpoint['optimizer_state'])
epoch = checkpoint['epoch']
loss = checkpoint['loss']